# 09 – Ciclo Brayton de Turbina a Gás

O **ciclo Brayton** é a espinha dorsal termodinâmica dos motores de turbina a
gás para geração de potência e propulsão a jato. A idealização padrão-ar
compreende quatro processos:

1. **Compressão isentrópica** (compressor): $P_1 \to P_2 = r_p P_1$
2. **Adição de calor isobárica** (câmara de combustão): $T_2 \to T_3$
3. **Expansão isentrópica** (turbina): $P_3 \to P_4 = P_3 / r_p$
4. **Rejeição de calor isobárica** (exaustão)

A análise de livros-texto assume $C_p$ e $\gamma = C_p/C_v$ constantes,
fornecendo a conhecida eficiência $\eta = 1 - r_p^{(1-\gamma)/\gamma}$. Aqui
usamos o $C_p(T)$ e $S^\circ(T)$ **reais** do `pyglenn` — o fluido de trabalho
é uma mistura de gás real de O₂ e N₂ (ar) — e resolvemos numericamente a
condição isentrópica $S(T_2) = S(T_1)$.

A eficiência térmica segue diretamente dos valores de entalpia em cada estado:

$$\eta = 1 - \frac{H_4 - H_1}{H_3 - H_2}.$$

In [ ]:
from pyglenn import ThermochemicalCalculator, R

_INDEX = {}

def species_id(calc, name):
    """Retorna o id no banco de dados da espécie cujo *nome* corresponde exatamente.

    ``get_available_species`` faz correspondência de substrings tanto no nome
    quanto na fórmula e limita o resultado a 20 linhas, de modo que nomes
    curtos como ``"O2"`` podem ser sufocados por entradas como ``"AL2O2"`` ou
    ``"CO2"``. Para sermos robustos, construímos um índice completo nome -> id
    uma única vez (armazenado em cache durante a sessão) e buscamos o nome
    exato nele.
    """
    if not _INDEX:
        _INDEX.update({s["name"]: s["id"] for s in calc.get_available_species("")})
    if name not in _INDEX:
        raise ValueError(f"Espécie {name!r} não encontrada no banco de dados")
    return _INDEX[name]

print("Constante universal dos gases R =", R, "J/(mol.K)")


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

from scipy.optimize import brentq

## 1. Ar como fluido de trabalho

Modelamos o ar seco como a mistura $1\ \mathrm{O_2} + 3{,}76\ \mathrm{N_2}$.
Funções para entalpia e entropia da mistura para uma única espécie seguem
abaixo.

In [ ]:
AIR = {"O2": 1.0, "N2": 3.76}
R_UNIV = R  # J/(mol·K) — constante universal dos gases

def air_enthalpy(calc, T):
    """Entalpia padronizada total de 1 mol de ar em T, em J."""
    return sum(n * calc.calculate_properties(species_id(calc, name), T)["h_relative"]
               for name, n in AIR.items())

def air_entropy_std(calc, T):
    """Entropia no estado padrão (P=1 bar) de 1 mol de ar em T, em J/K."""
    return sum(n * calc.calculate_properties(species_id(calc, name), T)["s"]
               for name, n in AIR.items())

# Total de mols na mistura por mol de "ar"
AIR_MOLES = sum(AIR.values())  # = 4.76


## 2. Relação isentrópica com $C_p(T)$ real

Para um gás ideal, a entropia molar em $(T,P)$ é

$$s(T,P) = s^\circ(T) - R \ln\frac{P}{P_0}.$$

Para um processo **isentrópico** ($s_2 = s_1$) entre as pressões $P_1$ e $P_2$,
os termos de pressão fornecem $s^\circ(T_2) - s^\circ(T_1) = R\ln(P_2/P_1)$.
Resolvemos

$$f(T) = s^\circ(T) - s^\circ(T_1) - R \ln\frac{P_2}{P_1} = 0$$

para $T_2$ com um buscador de raiz robusto com intervalo, contabilizando os mols
totais na mistura de ar.

In [ ]:
def isentropic_T(calc, T1, P_ratio):
    """Temperatura após compressão/expansão isentrópica (P2/P1 = P_ratio).

    Retorna T2 [K] tal que s°(T2) - s°(T1) = R_ar * ln(P2/P1),
    onde R_ar = R × (mols totais da mistura).
    """
    s1 = air_entropy_std(calc, T1)
    R_air = R_UNIV * AIR_MOLES  # J/K por mol da mistura de ar
    target = s1 + R_air * np.log(P_ratio)

    def f(T):
        return air_entropy_std(calc, T) - target

    # Intervalo de busca: entropia cresce com T, logo compressão (P_ratio>1)
    # empurra T2 > T1; expansão (P_ratio<1) empurra T2 < T1.
    if P_ratio > 1:
        lo, hi = T1, T1 * (P_ratio ** 0.4 + 1)
    else:
        lo, hi = T1 * P_ratio ** 0.4, T1

    # Expande o intervalo se necessário
    while f(lo) * f(hi) > 0:
        if P_ratio > 1:
            hi *= 1.3
        else:
            lo *= 0.7
        if hi > 6000 or lo < 50:
            raise RuntimeError(f"Não foi possível delimitar T2 para P_ratio={P_ratio}")

    return brentq(f, lo, hi)


## 3. Construindo o ciclo Brayton

Definimos os parâmetros do ciclo:
- Entrada: $T_1 = 300\ \mathrm{K}$, $P_1 = 1\ \mathrm{bar}$
- Razão de pressão: $r_p = P_2/P_1 = 10$
- Temperatura de entrada da turbina (TIT): $T_3 = 1400\ \mathrm{K}$

Os quatro estados são calculados com $C_p(T)$ real via o solver isentrópico.

In [ ]:
def brayton_states(calc, T1, T3, rp):
    """Retorna dict de (T, h_por_kg_ar) para os quatro estados Brayton.

    Os valores de h estão em kJ por kg de ar (não por mol da mistura).
    """
    # Massa molecular da mistura de ar (média ponderada por mol)
    M_O2 = 31.9988   # g/mol
    M_N2 = 28.0134   # g/mol
    M_air = (AIR["O2"] * M_O2 + AIR["N2"] * M_N2) / AIR_MOLES  # g/mol

    T2 = isentropic_T(calc, T1, rp)          # saída do compressor
    T4 = isentropic_T(calc, T3, 1.0 / rp)    # saída da turbina

    def h_kJ_kg(T):
        return air_enthalpy(calc, T) / (M_air * 1e-3) / 1000  # kJ/kg

    return {
        "1 (entrada do compressor)":  (T1, h_kJ_kg(T1)),
        "2 (saída do compressor)":    (T2, h_kJ_kg(T2)),
        "3 (entrada da turbina)":     (T3, h_kJ_kg(T3)),
        "4 (saída da turbina)":       (T4, h_kJ_kg(T4)),
    }

with ThermochemicalCalculator() as calc:
    states = brayton_states(calc, T1=300.0, T3=1400.0, rp=10)

print(f"{'Estado':<28s} {'T [K]':>8s} {'h [kJ/kg]':>12s}")
print("-" * 48)
for label, (T, h) in states.items():
    print(f"{label:<28s} {T:8.1f} {h:12.2f}")


## 4. Eficiência térmica

$$\eta_\mathrm{th} = \frac{w_\mathrm{líquido}}{q_\mathrm{entra}}
  = \frac{(h_3 - h_4) - (h_2 - h_1)}{h_3 - h_2}
  = 1 - \frac{h_4 - h_1}{h_3 - h_2}.$$

Comparamos o resultado com gás real e a fórmula de livro-texto com $\gamma$
constante ($\gamma \approx 1{,}4$ para ar diatômico).

In [ ]:
T1, h1 = states["1 (entrada do compressor)"]
T2, h2 = states["2 (saída do compressor)"]
T3, h3 = states["3 (entrada da turbina)"]
T4, h4 = states["4 (saída da turbina)"]

w_comp = h2 - h1          # trabalho do compressor [kJ/kg]
w_turb = h3 - h4          # trabalho da turbina [kJ/kg]
q_in   = h3 - h2          # calor adicionado [kJ/kg]
eta_real = (w_turb - w_comp) / q_in

# Padrão-ar frio (gamma constante = 1,4)
gamma = 1.4
rp = 10
eta_cold = 1 - rp ** ((1 - gamma) / gamma)

print(f"Eficiência Brayton gás real    : {eta_real:.4f} ({eta_real*100:.1f}%)")
print(f"Padrão-ar frio (gamma=1,4)      : {eta_cold:.4f} ({eta_cold*100:.1f}%)")
print(f"Trabalho da turbina  : {w_turb:.1f} kJ/kg")
print(f"Trabalho do compressor: {w_comp:.1f} kJ/kg")
print(f"Trabalho líquido      : {w_turb - w_comp:.1f} kJ/kg")


## 5. Diagrama T–s (qualitativo)

Esboçamos os quatro estados nos eixos temperatura–entropia, usando a entropia
padrão da mistura de ar na pressão de cada estado.

In [ ]:
def air_s_at_P(calc, T, P_bar):
    """Entropia da mistura de ar em (T, P), em kJ/(kg·K)."""
    s_std = air_entropy_std(calc, T)           # J/K por mol de mistura
    R_air = R_UNIV * AIR_MOLES                  # J/K por mol de mistura
    s_p = s_std - R_air * np.log(P_bar / 1.0)  # J/K por mol de mistura
    M_air = (AIR["O2"] * 31.9988 + AIR["N2"] * 28.0134) / AIR_MOLES
    return s_p / (M_air * 1e-3) / 1000          # kJ/(kg·K)

rp = 10
T1_val, T3_val = 300.0, 1400.0

with ThermochemicalCalculator() as calc:
    T2_val = isentropic_T(calc, T1_val, rp)
    T4_val = isentropic_T(calc, T3_val, 1.0 / rp)

    # Isobáricas em P=1 bar e P=10 bar
    T_range = np.linspace(250, 1600, 100)
    s_low   = np.array([air_s_at_P(calc, T, 1)  for T in T_range])
    s_high  = np.array([air_s_at_P(calc, T, 10) for T in T_range])

    # Os quatro estados
    s1 = air_s_at_P(calc, T1_val, 1)
    s2 = air_s_at_P(calc, T2_val, 10)
    s3 = air_s_at_P(calc, T3_val, 10)
    s4 = air_s_at_P(calc, T4_val, 1)

fig, ax = plt.subplots()
ax.plot(s_low,  T_range, "C0", lw=0.8, label="P = 1 bar")
ax.plot(s_high, T_range, "C1", lw=0.8, label="P = 10 bar")
# Setas dos processos
ax.annotate("", xy=(s2, T2_val), xytext=(s1, T1_val),
            arrowprops=dict(arrowstyle="->", color="C2", lw=2))
ax.annotate("", xy=(s3, T3_val), xytext=(s2, T2_val),
            arrowprops=dict(arrowstyle="->", color="C3", lw=2))
ax.annotate("", xy=(s4, T4_val), xytext=(s3, T3_val),
            arrowprops=dict(arrowstyle="->", color="C2", lw=2))
ax.annotate("", xy=(s1, T1_val), xytext=(s4, T4_val),
            arrowprops=dict(arrowstyle="->", color="C3", lw=2))
ax.scatter([s1, s2, s3, s4], [T1_val, T2_val, T3_val, T4_val],
           color="black", zorder=5)
for i, (s, T, lbl) in enumerate([(s1, T1_val, "1"), (s2, T2_val, "2"),
                                  (s3, T3_val, "3"), (s4, T4_val, "4")]):
    ax.text(s + 0.02, T + 10, lbl, fontweight="bold")
ax.set_xlabel("Entropia [kJ/(kg·K)]")
ax.set_ylabel("Temperatura [K]")
ax.set_title("Ciclo Brayton — ar com $C_p(T)$ real")
ax.legend()
plt.show()


## 6. Eficiência vs. razão de pressão: real vs. padrão-ar frio

A fórmula do padrão-ar frio $\eta = 1 - r_p^{(1-\gamma)/\gamma}$ prevê
eficiência monotonicamente crescente. Com $C_p(T)$ real, a curva de eficiência
tem forma similar, mas é deslocada porque $\gamma(T)$ diminui à medida que a
temperatura sobe (os modos vibracionais são ativados), tornando o compressor
"mais rígido" e a turbina menos efetiva em razões de pressão mais altas do que a
aproximação de ar frio sugere.

In [ ]:
rp_range = np.array([4, 6, 8, 10, 12, 15, 18, 20, 25, 30])
eta_real_list = []
eta_cold_list = []
backwork_list = []

with ThermochemicalCalculator() as calc:
    for rp in rp_range:
        st = brayton_states(calc, T1=300.0, T3=1400.0, rp=rp)
        _, h1 = st["1 (entrada do compressor)"]
        _, h2 = st["2 (saída do compressor)"]
        _, h3 = st["3 (entrada da turbina)"]
        _, h4 = st["4 (saída da turbina)"]
        w_comp = h2 - h1
        w_turb = h3 - h4
        q_in   = h3 - h2
        eta_real_list.append((w_turb - w_comp) / q_in * 100)
        eta_cold_list.append((1 - rp ** ((1 - 1.4) / 1.4)) * 100)
        backwork_list.append(w_comp / w_turb * 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))

ax1.plot(rp_range, eta_real_list, "o-", label="Real $C_p(T)$")
ax1.plot(rp_range, eta_cold_list, "s--", label=r"Ar frio ($\gamma$=1,4)")
ax1.set_xlabel("Razão de pressão $r_p$")
ax1.set_ylabel("Eficiência térmica [%]")
ax1.set_title("Eficiência Brayton: real vs. padrão-ar frio")
ax1.legend()

ax2.plot(rp_range, backwork_list, "o-", color="C2")
ax2.set_xlabel("Razão de pressão $r_p$")
ax2.set_ylabel("Razão de trabalho reverso $w_c / w_t$ [%]")
ax2.set_title("Compressor consome uma fração crescente do trabalho da turbina")

plt.tight_layout()
plt.show()


## Resumo

- $C_p(T)$ real do `pyglenn` permite análise precisa do ciclo Brayton sem
  assumir $\gamma$ constante.
- A condição isentrópica $s^\circ(T_2) = s^\circ(T_1) + R\ln(P_2/P_1)$ é
  resolvida numericamente para cada processo.
- A curva de eficiência real é próxima, mas distinta, do ideal de ar frio; a
  diferença cresce em razões de pressão mais altas, onde $\gamma(T)$ se desvia
  de 1,4.
- A razão de trabalho reverso (trabalho do compressor / trabalho da turbina)
  cresce com $r_p$, limitando as razões de pressão práticas para motores de
  eixo único.

**A seguir:** o notebook 10 mostra como usar o `pyglenn` como um provedor de
propriedades de alto desempenho para códigos de CFD e cinética química.